# 01 — Labels and Schema: TK/BC Labels as First-Class Objects

**Repo:** local_contexts_geospatial  
**Author:** Lilly Jones, PhD — Daear Consulting, LLC  

---

## What This Notebook Covers

Local Contexts Traditional Knowledge (TK) and Biocultural (BC) labels
are designed to travel with data — signaling cultural authority,
appropriate use, and community expectations across the full data lifecycle.

This notebook introduces the `localcontexts` toolkit's label schema:

1. What TK and BC labels are and when each type applies
2. The `TKMetadata` and `BCMetadata` objects — labels as structured data
3. How to serialize labels to JSON, attach them to metadata dicts,
   and reconstruct them from stored data
4. The full TK and BC label vocabulary

## The Core Shift

> **From:** a label as a note in a README  
> **To:** a label as a structured object embedded in the data itself

When a label lives in a README, it is documentation. When it lives
in the data's metadata — in a GeoTIFF tag, a GeoJSON property, an
xarray attribute, or a STAC extension — it can be read, validated,
and enforced programmatically. It travels with the data instead of
living in a separate file that gets separated from the data it describes.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json

from localcontexts.labels import (
    TKLabel, BCLabel,
    TKMetadata, BCMetadata,
    TK_LABEL_DESCRIPTIONS,
    has_tk_label, has_bc_label, has_any_label,
    extract_tk_fields, extract_bc_fields,
)

print("localcontexts toolkit loaded.")
print(f"TK label types available: {len(TKLabel)}")
print(f"BC label types available: {len(BCLabel)}")

---
## Part 1: The TK and BC Label Vocabulary

Local Contexts provides two families of labels. Understanding when
to use each is essential before attaching any label to data.

In [ ]:
# Print the full TK label vocabulary
print("TRADITIONAL KNOWLEDGE (TK) LABELS")
print("Designed for cultural heritage materials — recordings, images,")
print("documents, and data about Indigenous knowledge and practices.")
print("=" * 60)
for label in TKLabel:
    print(f"  {label.value}")

In [ ]:
# Print the full BC label vocabulary
print("BIOCULTURAL (BC) LABELS")
print("Designed for biological diversity and genetic resource data —")
print("plant, animal, and ecosystem knowledge tied to Indigenous territories.")
print("=" * 60)
for label in BCLabel:
    print(f"  {label.value}")

In [ ]:
# When to use TK vs BC
print("TK vs. BC: Which label type applies?")
print("=" * 60)
print()
print("Use TK labels for data about:")
print("  - Cultural practices, ceremonies, and oral traditions")
print("  - Traditional land use and management knowledge")
print("  - Historical photographs, recordings, and documents")
print("  - Environmental knowledge embedded in cultural practice")
print("  - Geospatial data derived from or describing cultural sites")
print()
print("Use BC labels for data about:")
print("  - Species occurrence and distribution in Indigenous territories")
print("  - Plant and animal knowledge tied to specific communities")
print("  - Genetic resource data from Indigenous lands")
print("  - Biodiversity monitoring on Tribal lands")
print("  - Ecosystem services data with community stewardship context")
print()
print("Some datasets warrant both a TK and a BC label — for example,")
print("a dataset of traditional medicinal plant locations that carries")
print("both cultural knowledge and biological resource dimensions.")

---
## Part 2: TKMetadata — Labels as Structured Objects

In [ ]:
# Create a TK label for a hypothetical vegetation survey dataset
# that includes observations tied to traditional land knowledge

tk = TKMetadata(
    label     = TKLabel.NON_COMMERCIAL,
    community = "Oglala Lakota Nation",
    authority = "Tribal Data Governance Office",
    usage     = (
        "Non-commercial environmental research only. "
        "Results must be shared with the Tribal Data Governance Office "
        "before publication."
    ),
    contact   = "data@oglalalakota.org",
    notes     = (
        "This dataset describes vegetation conditions on Pine Ridge. "
        "Traditional land knowledge informed the site selection. "
        "Derived datasets must retain this label."
    ),
)

print("TKMetadata object created:")
print(tk)

In [ ]:
# The label description — what this label means in plain language
print("Label description:")
print("-" * 50)
print(tk.describe())

In [ ]:
# Attach the label to a metadata dict
# This is the operation that embeds the label in your data

dataset_meta = {
    "name":     "pine_ridge_vegetation_survey",
    "source":   "Field survey 2023",
    "crs":      "EPSG:4326",
    "n_sites":  47,
}

# .attach() returns a NEW dict — does not modify the original
labeled_meta = tk.attach(dataset_meta)

print("Dataset metadata BEFORE attaching label:")
print(json.dumps(dataset_meta, indent=2))
print()
print("Dataset metadata AFTER attaching label:")
print(json.dumps(labeled_meta, indent=2))

In [ ]:
# Helper functions for inspecting labeled metadata

print(f"Has TK label: {has_tk_label(labeled_meta)}")
print(f"Has BC label: {has_bc_label(labeled_meta)}")
print(f"Has any label: {has_any_label(labeled_meta)}")
print()
print("TK fields only:")
print(json.dumps(extract_tk_fields(labeled_meta), indent=2))

In [ ]:
# Round-trip: serialize to JSON and reconstruct
# This proves the label survives storage and retrieval

# Serialize
json_str = tk.to_json()
print("Serialized TKMetadata JSON:")
print(json_str)

# Reconstruct from the tk: prefixed fields in labeled_meta
tk_fields = extract_tk_fields(labeled_meta)
reconstructed = TKMetadata.from_dict(tk_fields)

print()
print("Reconstructed from metadata dict:")
print(f"  Label     : {reconstructed.label.value}")
print(f"  Community : {reconstructed.community}")
print(f"  Authority : {reconstructed.authority}")
print(f"  Match     : {reconstructed.label == tk.label and reconstructed.community == tk.community}")

---
## Part 3: BCMetadata — Biocultural Labels

In [ ]:
# Create a BC label for a species occurrence dataset

bc = BCMetadata(
    label     = BCLabel.NON_COMMERCIAL,
    community = "Rosebud Sioux Tribe",
    authority = "Rosebud Sioux Tribe Natural Resources Department",
    usage     = (
        "Non-commercial research only. Species location data may not "
        "be used for commercial bioprospecting or genetic resource "
        "extraction without explicit Tribal consent."
    ),
    species   = "Native prairie grasses — Bouteloua gracilis, Pascopyrum smithii",
    territory = "Rosebud Reservation, South Dakota",
    contact   = "naturalresources@rosebudsiouxtribe.org",
)

print("BCMetadata object:")
print(bc)

In [ ]:
# Datasets can carry both TK and BC labels
# A traditional plant knowledge dataset, for example

plant_knowledge_meta = {"name": "rosebud_traditional_plants"}

tk_plant = TKMetadata(
    label     = TKLabel.CULTURALLY_SENSITIVE,
    community = "Rosebud Sioux Tribe",
    authority = "Rosebud Sioux Tribe Cultural Preservation Office",
    usage     = "Restricted — community use only. Not for external publication.",
)

# Attach both labels
plant_knowledge_meta = tk_plant.attach(plant_knowledge_meta)
plant_knowledge_meta = bc.attach(plant_knowledge_meta)

print("Dataset with both TK and BC labels:")
print(json.dumps(plant_knowledge_meta, indent=2))
print()
print(f"Has TK label: {has_tk_label(plant_knowledge_meta)}")
print(f"Has BC label: {has_bc_label(plant_knowledge_meta)}")

---
## Part 4: Label Registry and Discovery

In [ ]:
# Print descriptions for all labels that have them in the registry
print("TK LABEL DESCRIPTIONS (subset)")
print("=" * 60)
for label, description in list(TK_LABEL_DESCRIPTIONS.items())[:4]:
    print(f"\n{label.value}:")
    # Wrap text
    words, line = description.split(), ""
    for word in words:
        if len(line) + len(word) + 1 > 65:
            print(f"  {line}")
            line = word
        else:
            line = f"{line} {word}".strip()
    if line:
        print(f"  {line}")

print()
print(f"See localcontexts/labels.py for all {len(TK_LABEL_DESCRIPTIONS)} descriptions.")
print("Full label reference: https://localcontexts.org/labels/traditional-knowledge-labels/")

---
## Summary

| Object | What it represents | Key method |
|---|---|---|
| `TKLabel` | A TK label type from the LC vocabulary | `.value` |
| `BCLabel` | A BC label type from the LC vocabulary | `.value` |
| `TKMetadata` | A TK label with community, authority, usage | `.attach(meta)` |
| `BCMetadata` | A BC label with community, authority, usage | `.attach(meta)` |
| `extract_tk_fields(meta)` | Pull just tk: fields from any dict | — |
| `TKMetadata.from_dict(d)` | Reconstruct from stored metadata | — |

---
## Discussion Questions

1. A researcher is building a dataset that combines MODIS satellite
   imagery (public federal data) with GPS waypoints collected by
   Tribal range riders during pasture condition surveys. Which part
   of the combined dataset warrants a TK label? Should the satellite
   data portion also carry the label?

2. The `TK Culturally Sensitive` and `TK Community Use Only` labels
   both restrict external access. When would you use one vs. the other?

3. The `TK Notice` label is designed for situations where Indigenous
   interests exist in a dataset but a specific label hasn't been
   assigned yet. Why might a researcher want to apply a Notice label
   proactively, before the community has assigned a more specific label?

---
## Next Notebook

**02 — Vector Workflows:** Attaching TK/BC labels to GeoJSON features
and GeoPackage attributes so they travel with vector geospatial data.